# FUNDAMENTO TEÓRICO

## 1. Definición Formal de un DFA

Un **Autómata Finito Determinista (DFA)** se define formalmente como una 5-tupla:

$$M = (Q, \Sigma, \delta, q_0, F)$$

Donde:
1. **$Q$**: Conjunto finito de **estados**.
2. **$\Sigma$**: **Alfabeto** finito de entrada.
3. **$\delta: Q \times \Sigma \to Q$**: **Función de transición**.
4. **$q_0 \in Q$**: **Estado inicial**
5. **$F \subseteq Q$**: Conjunto de **estados finales**.

Un lenguaje $L(M)$ es el **subconjunto** de $\Sigma*$ que acepta $M$.

### *Teorema: Todo lenguaje que reconoce un DFA es regular y para todo lenguaje regular existe un DFA que lo reconoce.*

## 2. Clausuras en los Lenguajes Regulares

Los **Lenguajes Regulares** son **cerrados** bajo el **complemento**, la **intersección** y la **unión**.

## 3. Demostración de Equivalencia mediante la Diferencia Simétrica

Para comprobar si dos autómatas son equivalentes ($L(M_1) = L(M_2)$), recurrimos a la **Diferencia Simétrica** de conjuntos:

$$L_{\text{dif}} = (L(M_1) \cap \overline{L(M_2)}) \cup (\overline{L(M_1)} \cap L(M_2))$$

- $L_{\text{dif}}$ contiene todas las cadenas que son aceptadas por $M_1$ pero NO por $M_2$, o viceversa.
- Si $L(M_1) = L(M_2)$, la diferencia simétrica debe ser un conjunto **vacío** ($L_{\text{dif}} = \emptyset$).
- Si $L(M_1) \neq L(M_2)$, existirán cadenas en $L_{\text{dif}}$ que sirven como **contraejemplos** de la equivalencia.

Dado que tenemos implementadas las funciones de *Complemento*, *Intersección* y *Unión*, podemos construir el DFA resultante de $L_{\text{dif}}$ de forma totalmente automatizada.

---

## 4. BFS (Breadth-First Search)

Una vez obtenido el DFA de la diferencia simétrica, el problema de equivalencia se reduce a responder: **¿Es $L(DFA_{\text{dif}}) = \emptyset$?**

Esto es equivalente a responder: **¿Existe algún camino en el grafo del autómata desde el estado inicial hasta algún estado de aceptación?**

### ¿Por qué usamos BFS?
1. **Modelado como Grafo:** Un DFA es estructuralmente un grafo dirigido donde los nodos son los estados y las aristas son las transiciones etiquetadas con símbolos del alfabeto.
2. **Garantía de Cadena Mínima:** BFS explora el grafo nivel por nivel desde el estado inicial. Si existe al menos un estado final alcanzable, BFS garantiza encontrar **el camino más corto** hasta él.
3. **Decidibilidad:** Como el número de estados en el autómata producto es finito, BFS siempre se detendrá (ya sea encontrando un estado final o habiendo visitado todos los estados alcanzables).

### Interpretación del Resultado:
- **Si BFS NO encuentra ningún estado final:** No hay cadenas que diferencien a los autómatas. **Resultado: "Son equivalentes"**.
- **Si BFS ENCUENTRA un estado final:** Los símbolos recopilados a lo largo de las aristas del camino forman la cadena contraejemplo. **Resultado: "No son equivalentes" + Cadena contraejemplo**.

# ENTRADA

Para representar la $5$-tupla de un DFA en nuestro entorno de ejecución, utilizamos archivos con formato `.csv`, los cuales cargamos como `DataFrames` en Pandas. Esta estructura tabular mapea de forma directa la tabla de transiciones y los metadatos del autómata.

**Archivos que utilizamos:**


DFA1:
Acepta cualquier cadena que termine en 1
```csv
Estado,Estado Inicial,Estados Finales,0,1
q0,x,,q0,q1
q1,,x,q0,q1
```
DFA2:
Acepta cualquier cadena que termine en 1

```csv
Estado,Estado Inicial,Estados Finales,0,1
q0,x,,q1,q2
q1,,,q1,q2
q2,,x,q1,q2
```
DFA3:
Acepta cualquier cadena con un número impar de 0s.

```csv
Estado,Estado Inicial,Estados Finales,0,1
q0,x,,q1,q0
q1,,x,q0,q1
```

Note que el DFA1 y DFA2 aceptan el mismo lenguaje.

In [9]:
import pandas as pd

M1 = pd.read_csv('DFA1.csv')
M2 = pd.read_csv('DFA2.csv')

# PASO 1

Nota: En lugar de definir una función para la unión, aprovechamos la siguiente propiedad  de **De Morgan**:

$$L(M_1) \cup L(M_2) = \overline{\overline{L(M_1)} \cap \overline{L(M_2)}}$$

In [5]:
def complementar_dfa(df):
    df_comp = df.copy(deep=True)
    df_comp['Estados Finales'] = df_comp['Estados Finales'].apply(lambda val: None if val == 'x' else 'x')
    return df_comp

def intersectar_dfas(df1, df2):
    columnas_control = ['Estado Inicial', 'Estados Finales']
    alfabeto = [col for col in df1.columns if col not in columnas_control]

    nuevas_filas = []

    for q1 in df1.index:
        for q2 in df2.index:
            nuevo_estado = f"({q1},{q2})"

            es_inicial = 'x' if (df1.loc[q1, 'Estado Inicial'] == 'x' and df2.loc[q2, 'Estado Inicial'] == 'x') else None

            es_final = 'x' if (df1.loc[q1, 'Estados Finales'] == 'x' and df2.loc[q2, 'Estados Finales'] == 'x') else None

            fila = {
                'Estado': nuevo_estado,
                'Estado Inicial': es_inicial,
                'Estados Finales': es_final
            }

            for simbolo in alfabeto:
                dest1 = df1.loc[q1, simbolo]
                dest2 = df2.loc[q2, simbolo]
                fila[simbolo] = f"({dest1},{dest2})"

            nuevas_filas.append(fila)

    return pd.DataFrame(nuevas_filas).set_index('Estado')

def unir_dfas(df1, df2):
    comp1 = complementar_dfa(df1)
    comp2 = complementar_dfa(df2)
    intersec = intersectar_dfas(comp1, comp2)
    return complementar_dfa(intersec)

In [10]:
dfa = unir_dfas(intersectar_dfas(M1, complementar_dfa(M2)),intersectar_dfas(complementar_dfa(M1),M2))

dfa

,Estado Inicial,Estados Finales,0,1
Estado,,,,
"((q0,q0),(q0,q0))",x,None,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q0),(q0,q1))",None,None,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q0),(q0,q2))",None,x,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q0),(q1,q0))",None,None,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q0),(q1,q1))",None,None,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q0),(q1,q2))",None,None,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q1),(q0,q0))",None,None,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q1),(q0,q1))",None,None,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"
"((q0,q1),(q0,q2))",None,x,"((q0,q1),(q0,q1))","((q1,q2),(q1,q2))"


# PASO 2 y 3

In [7]:
from collections import deque

def E_DFA(df):

    columnas_control = ['Estado Inicial', 'Estados Finales']
    alfabeto = [col for col in df.columns if col not in columnas_control]

    estados_iniciales = df[df['Estado Inicial'] == 'x'].index

    estado_inicial = estados_iniciales[0]

    if df.loc[estado_inicial, 'Estados Finales'] == 'x':
        return "No son equivalentes", ""

    cola = deque([(estado_inicial, "")])
    visitados = {estado_inicial}

    while cola:
        estado_actual, camino = cola.popleft()

        for simbolo in alfabeto:
            siguiente_estado = df.loc[estado_actual, simbolo]

            nuevo_camino = camino + str(simbolo)

            if df.loc[siguiente_estado, 'Estados Finales'] == 'x':
                return "No son equivalentes", nuevo_camino

            if siguiente_estado not in visitados:
                visitados.add(siguiente_estado)
                cola.append((siguiente_estado, nuevo_camino))

    return "Son equivalentes", None

# SALIDA

In [11]:
print(E_DFA(dfa))

('Son equivalentes', None)


# CONCLUSIÓN
Este ejercicio me pareció bastante enriquecedor. Fue muy bacano ver cómo la teoría de autómatas se conecta directamente con el código al representar los DFA como una estructura de dato y usar BFS para verificar la vaciedad del autómata de la diferencia simétrica. Me llamó mucho la atención comprobar en la práctica la decidibilidad del problema ($EQ_{\text{DFA}}$), ya que el algoritmo siempre termina en tiempo polinomial $\mathcal{O}(\vert{}Q_1\vert{} \times \vert{}Q_2\vert{} \times \vert{}\Sigma\vert{})$ debido al tamaño del autómata producto, esto garantiza que nos da una respuesta determinista y devolviendo la cadena contraejemplo más corta posible cuando los lenguajes no coinciden.